# Exercice 19 — Pipeline complet sur données capteurs

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

# Reproductibilité
rng = np.random.default_rng(42)

# 500 relevés horaires répartis sur 3 semaines et 3 capteurs
n = 500
start = pd.Timestamp("2026-10-01 00:00:00")
hours = rng.integers(0, 21 * 24, size=n)

df = pd.DataFrame({
    "date_heure": start + pd.to_timedelta(hours, unit="h"),
    "capteur": rng.choice(["CAP_01", "CAP_02", "CAP_03"], size=n, p=[0.34, 0.33, 0.33])
})

# Mesures simulées avec des profils légèrement différents selon le capteur
sensor_params = {
    "CAP_01": {"temp": 25.0, "hum": 62.0, "press": 1012.0},
    "CAP_02": {"temp": 27.0, "hum": 58.0, "press": 1010.0},
    "CAP_03": {"temp": 23.0, "hum": 68.0, "press": 1014.0},
}

df["temperature"] = [
    rng.normal(sensor_params[s]["temp"], 2.0) for s in df["capteur"]
]
df["humidite"] = [
    rng.normal(sensor_params[s]["hum"], 6.0) for s in df["capteur"]
]
df["pression"] = [
    rng.normal(sensor_params[s]["press"], 3.0) for s in df["capteur"]
]

# Injection de valeurs aberrantes dans la température
outlier_idx = rng.choice(df.index, size=15, replace=False)
df.loc[outlier_idx, "temperature"] += rng.choice([-1, 1], size=15) * rng.uniform(12, 20, size=15)

# Injection de valeurs manquantes dans l'humidité
missing_idx = rng.choice(df.index, size=40, replace=False)
df.loc[missing_idx, "humidite"] = np.nan

# Détection des anomalies par la méthode IQR, séparément pour chaque capteur
df["temperature_anormale"] = False
df["temperature_nettoyee"] = df["temperature"]

for capteur, indices in df.groupby("capteur").groups.items():
    serie = df.loc[indices, "temperature"]
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    borne_basse = q1 - 1.5 * iqr
    borne_haute = q3 + 1.5 * iqr

    anomalies = (serie < borne_basse) | (serie > borne_haute)
    df.loc[indices, "temperature_anormale"] = anomalies
    # Neutralisation par winsorisation : remplacement par la borne la plus proche
    df.loc[indices, "temperature_nettoyee"] = serie.clip(
        lower=borne_basse, upper=borne_haute
    )

# Imputation de l'humidité par la médiane du capteur correspondant
df["humidite"] = df.groupby("capteur")["humidite"].transform(
    lambda s: s.fillna(s.median())
)

# Score Z calculé sur les températures nettoyées, par capteur
groupes_temp = df.groupby("capteur")["temperature_nettoyee"]
moyennes = groupes_temp.transform("mean")
ecarts_types = groupes_temp.transform("std").replace(0, np.nan)
df["score_z_temperature"] = (df["temperature_nettoyee"] - moyennes) / ecarts_types

# Agrégation quotidienne par capteur
df["date"] = df["date_heure"].dt.date
resultat = (
    df.groupby(["capteur", "date"], as_index=False)
      .agg(
          temperature_moyenne=("temperature_nettoyee", "mean"),
          humidite_moyenne=("humidite", "mean"),
          pression_moyenne=("pression", "mean"),
          score_z_temperature_moyen=("score_z_temperature", "mean"),
          nombre_releves=("temperature", "size"),
          nombre_anomalies_temperature=("temperature_anormale", "sum")
      )
)

# Export CSV
output_path = Path("/home/billa/IA & ML/CORRIGES/1. Numpy & pandas/ex19.csv")
resultat.to_csv(output_path, index=False, encoding="utf-8")

print(f"Fichier créé : {output_path}")
print(f"Nombre de relevés simulés : {len(df)}")
print(f"Anomalies de température détectées : {int(df['temperature_anormale'].sum())}")
print(f"Valeurs d'humidité manquantes imputées : {len(missing_idx)}")
print(f"Lignes après agrégation capteur/jour : {len(resultat)}")
print("\nAperçu du résultat :")
print(resultat.head(10).to_string(index=False))


Fichier créé : /home/billa/IA & ML/CORRIGES/1. Numpy & pandas/ex19.csv
Nombre de relevés simulés : 500
Anomalies de température détectées : 21
Valeurs d'humidité manquantes imputées : 40
Lignes après agrégation capteur/jour : 63

Aperçu du résultat :
capteur       date  temperature_moyenne  humidite_moyenne  pression_moyenne  score_z_temperature_moyen  nombre_releves  nombre_anomalies_temperature
 CAP_01 2026-10-01            25.397309         66.669714       1011.466651                   0.158904               6                             0
 CAP_01 2026-10-02            23.198193         64.278584       1012.607587                  -0.822229               5                             1
 CAP_01 2026-10-03            24.857239         63.760917       1012.429462                  -0.082048              10                             0
 CAP_01 2026-10-04            26.278929         65.616085       1012.206906                   0.552238               5                             1
 CAP